In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
spark

In [3]:
df = spark.read.parquet('/content/drive/MyDrive/pyspark_training/yellow_tripdata_2026-01.parquet')
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2026-01-01 00:54:04|  2026-01-01 00:59:37|              1|         0.97|         1|                 N|         239|    

In [4]:
df.select('tpep_pickup_datetime', 'tpep_dropoff_datetime').show(5)

+--------------------+---------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|
+--------------------+---------------------+
| 2026-01-01 00:54:04|  2026-01-01 00:59:37|
| 2026-01-01 00:34:04|  2026-01-01 00:39:47|
| 2026-01-01 00:57:06|  2026-01-01 01:05:59|
| 2026-01-01 00:15:22|  2026-01-01 00:58:10|
| 2026-01-01 00:27:13|  2026-01-01 00:40:43|
+--------------------+---------------------+
only showing top 5 rows


In [5]:
from pyspark.sql.functions import unix_timestamp, round

In [9]:
df1 = df.withColumn(
    'trip_duration_minutes',
    round(
        (unix_timestamp('tpep_dropoff_datetime') - unix_timestamp('tpep_pickup_datetime')) / 60,
        1
    )
)

In [10]:
df1.select('tpep_pickup_datetime', 'tpep_dropoff_datetime', 'trip_duration_minutes').show(5)

+--------------------+---------------------+---------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_duration_minutes|
+--------------------+---------------------+---------------------+
| 2026-01-01 00:54:04|  2026-01-01 00:59:37|                  5.6|
| 2026-01-01 00:34:04|  2026-01-01 00:39:47|                  5.7|
| 2026-01-01 00:57:06|  2026-01-01 01:05:59|                  8.9|
| 2026-01-01 00:15:22|  2026-01-01 00:58:10|                 42.8|
| 2026-01-01 00:27:13|  2026-01-01 00:40:43|                 13.5|
+--------------------+---------------------+---------------------+
only showing top 5 rows


In [12]:
df2 = df1.select(
    'VendorID',
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'passenger_count',
    'trip_distance',
    'payment_type',
    'total_amount',
    'trip_duration_minutes'
).withColumnsRenamed({
    'VendorID': 'vendor_id',
    'tpep_pickup_datetime': 'pickup_time',
    'tpep_dropoff_datetime': 'dropoff_time',
    'passenger_count': 'num_passengers',
    'trip_distance': 'distance_km',
    'payment_type': 'payment_method',
    'total_amount': 'total_fare'
})

In [13]:
df2.show(5)

+---------+-------------------+-------------------+--------------+-----------+--------------+----------+---------------------+
|vendor_id|        pickup_time|       dropoff_time|num_passengers|distance_km|payment_method|total_fare|trip_duration_minutes|
+---------+-------------------+-------------------+--------------+-----------+--------------+----------+---------------------+
|        2|2026-01-01 00:54:04|2026-01-01 00:59:37|             1|       0.97|             1|     15.86|                  5.6|
|        1|2026-01-01 00:34:04|2026-01-01 00:39:47|             0|        0.9|             2|     13.65|                  5.7|
|        1|2026-01-01 00:57:06|2026-01-01 01:05:59|             0|        1.4|             1|     18.95|                  8.9|
|        2|2026-01-01 00:15:22|2026-01-01 00:58:10|             4|       5.58|             1|     55.56|                 42.8|
|        2|2026-01-01 00:27:13|2026-01-01 00:40:43|             0|       2.16|             1|      23.1|       

In [17]:
df2.drop('vendor_id', 'payment_method').show(5)

+-------------------+-------------------+--------------+-----------+----------+---------------------+
|        pickup_time|       dropoff_time|num_passengers|distance_km|total_fare|trip_duration_minutes|
+-------------------+-------------------+--------------+-----------+----------+---------------------+
|2026-01-01 00:54:04|2026-01-01 00:59:37|             1|       0.97|     15.86|                  5.6|
|2026-01-01 00:34:04|2026-01-01 00:39:47|             0|        0.9|     13.65|                  5.7|
|2026-01-01 00:57:06|2026-01-01 01:05:59|             0|        1.4|     18.95|                  8.9|
|2026-01-01 00:15:22|2026-01-01 00:58:10|             4|       5.58|     55.56|                 42.8|
|2026-01-01 00:27:13|2026-01-01 00:40:43|             0|       2.16|      23.1|                 13.5|
+-------------------+-------------------+--------------+-----------+----------+---------------------+
only showing top 5 rows
